In [1]:
# Import packages

import kagglehub
import os
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
#import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_validate, GridSearchCV, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
#import joblib
import lightgbm as lgb
#import mlflow
#import optuna


In [2]:
# Get project location
parent_dir = os.path.dirname(os.getcwd())

input_folder_dir = fr"{parent_dir}\input_files"
test_output_dir = fr"{parent_dir}\test_output"

# Create folders if they don't exist
if not os.path.exists(input_folder_dir):
    os.makedirs(input_folder_dir)

if not os.path.exists(test_output_dir):
    os.makedirs(test_output_dir)

    
# Download titanic competition dataset if not already downloaded

if not os.listdir(input_folder_dir):
    download_path = kagglehub.competition_download("titanic",output_dir=rf"{parent_dir}\input_files")

df = {}
df['train'] = pd.read_csv(os.path.join(input_folder_dir, "train.csv"))
df['test'] = pd.read_csv(os.path.join(input_folder_dir, "test.csv"))

In [3]:
# Describe train and test sets

for x in df.keys():
    print(f"{x} size: {df[x].shape}")
    
          

train size: (891, 12)
test size: (418, 11)


In [4]:
for x in df.keys():
    print(f"{x} dtypes: {df[x].dtypes}")
    

train dtypes: PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object
test dtypes: PassengerId      int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object


In [5]:
for x in df.keys():
    print(f"{x} summary: {df[x].describe()}")

train summary:        PassengerId    Survived      Pclass         Age       SibSp  \
count   891.000000  891.000000  891.000000  714.000000  891.000000   
mean    446.000000    0.383838    2.308642   29.699118    0.523008   
std     257.353842    0.486592    0.836071   14.526497    1.102743   
min       1.000000    0.000000    1.000000    0.420000    0.000000   
25%     223.500000    0.000000    2.000000   20.125000    0.000000   
50%     446.000000    0.000000    3.000000   28.000000    0.000000   
75%     668.500000    1.000000    3.000000   38.000000    1.000000   
max     891.000000    1.000000    3.000000   80.000000    8.000000   

            Parch        Fare  
count  891.000000  891.000000  
mean     0.381594   32.204208  
std      0.806057   49.693429  
min      0.000000    0.000000  
25%      0.000000    7.910400  
50%      0.000000   14.454200  
75%      0.000000   31.000000  
max      6.000000  512.329200  
test summary:        PassengerId      Pclass         Age       Sib

In [6]:
for x in df.keys():
    print(f"{x} head: {df[x].head()}")
    

train head:    PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN  

**Create a simple random forest classifier model as a baseline:**

In [7]:
# start with naive numeric-only model

X_train = df['train'].select_dtypes(exclude = ["object"]).drop(columns = ['Survived']).copy()
y_train = df['train']['Survived'].copy()
X_test = df['test'].select_dtypes(exclude = ["object"]).copy()

# use two metrics for now
metrics = ["accuracy", "roc_auc"]

# pipeline
initial_random_forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", SimpleImputer(strategy="median")),
        (
            "model",
            RandomForestClassifier(
                n_estimators=100, max_depth=5, random_state=1
            ),
        ),
    ]
)


In [8]:
# run 5 fold cv 
cv_results = cross_validate(
    initial_random_forest_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring=metrics,
    return_train_score=True,
)

print(f"Train set - Mean CV accuracy: {np.mean(cv_results['train_accuracy']):.4f}")
print(f"Validation set - Mean CV accuracy: {np.mean(cv_results['test_accuracy']):.4f}")

print(f"Train set - Mean AUC ROC accuracy: {np.mean(cv_results['train_roc_auc']):.4f}")
print(f"Validation set - Mean AUC ROC accuracy: {np.mean(cv_results['test_roc_auc']):.4f}")


Train set - Mean CV accuracy: 0.7834
Validation set - Mean CV accuracy: 0.7274
Train set - Mean AUC ROC accuracy: 0.8549
Validation set - Mean AUC ROC accuracy: 0.7524


In [9]:
# Initial submission

initial_random_forest_pipeline.fit(X_train, y_train)

final_preds = initial_random_forest_pipeline.predict(X_test)

submission = pd.DataFrame({"PassengerId": df["test"]["PassengerId"], "Survived": final_preds})


print(submission['Survived'].value_counts())

    
submission.to_csv(rf"{test_output_dir}/1_init_rf_submission.csv", index=False)


Survived
0    279
1    139
Name: count, dtype: int64


**Fairly poor result (0.687) Now try a grid search to improve:**

In [10]:
param_grid = {
    "model__n_estimators": [45, 50, 55],
    "model__max_depth": [4, 5, 6],
    "model__min_samples_split": [9, 10,11],
}

grid_search = GridSearchCV(
    estimator=initial_random_forest_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring={"accuracy": "accuracy","roc_auc": "roc_auc"},
    refit="roc_auc",
    return_train_score=True,
    verbose=1,
)

# run the grid search
grid_search.fit(X_train, y_train)

best_index = grid_search.best_index_
results = grid_search.cv_results_

print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Train set - Mean CV accuracy: {results['mean_train_accuracy'][best_index]:.4f}")
print(f"Validation set - Mean CV accuracy: {results['mean_test_accuracy'][best_index]:.4f}")
print(f"Train set - Mean AUC ROC accuracy: {results['mean_train_roc_auc'][best_index]:.4f}")
print(f"Validation set - Mean AUC ROC accuracy: {results['mean_test_roc_auc'][best_index]:.4f}")

# 
final_preds = grid_search.predict(X_test)
submission = pd.DataFrame({"PassengerId": df["test"]["PassengerId"], "Survived": final_preds})
print(submission['Survived'].value_counts())
submission.to_csv(rf"{test_output_dir}/2_cvgrid_rf_submission.csv", index=False)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best Parameters Found: {'model__max_depth': 5, 'model__min_samples_split': 11, 'model__n_estimators': 55}
Train set - Mean CV accuracy: 0.7755
Validation set - Mean CV accuracy: 0.7251
Train set - Mean AUC ROC accuracy: 0.8438
Validation set - Mean AUC ROC accuracy: 0.7562
Survived
0    291
1    127
Name: count, dtype: int64


**Extremely slight improvement: 0.691, Some feature engineering, add categorical variables, imputation flags**

In [11]:
print(df['train'].columns)

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')


In [12]:
# Reload from raw
X_train = df['train'].copy()
y_train = df['train']['Survived'].copy()
X_test = df['test'].copy()

# flag for if age is estimated in given data
X_train['est_age_flag'] = np.where(X_train['Age'].notna() & (abs(X_train['Age'] - 0.5) - np.floor(X_train['Age'])) <0.1   , 1 , 0 )
X_test['est_age_flag'] = np.where(X_test['Age'].notna() & (abs(X_test['Age'] - 0.5) - np.floor(X_test['Age'])) <0.1 , 1 , 0 )

# flag for if age is imputed
X_train['imp_age_flag'] = np.where(X_train['Age'].isna(), 1 , 0 )
X_test['imp_age_flag'] = np.where(X_test['Age'].isna(), 1 , 0 )

# manually impute age
X_train['Age'] = X_train['Age'].fillna(X_train['Age'].median())
X_test['Age'] = X_test['Age'].fillna(X_train['Age'].median())

# Cabin letter (U if unknown)
X_train['cabin_letter'] = X_train['Cabin'].str[0].fillna("U")
X_test['cabin_letter'] = X_test['Cabin'].str[0].fillna("U")


# Manually impute embarked
X_train['Embarked'] = X_train['Embarked'].fillna("U")
X_test['Embarked'] = X_test['Embarked'].fillna("U")

# Drop unwanted columns
X_train = X_train.drop(columns = ['Name','Cabin','Survived',"Ticket","PassengerId"])
X_test = X_test.drop(columns = ['Name','Cabin',"Ticket","PassengerId"])

# define categorical features
numerical_features = X_train.select_dtypes(exclude=['object', 'string']).columns
categorical_features = X_train.select_dtypes(include=['object', 'string']).columns





In [13]:
print(f"Numerical features: {numerical_features}")
print(f"Categorical features: {categorical_features}")

Numerical features: Index(['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'est_age_flag',
       'imp_age_flag'],
      dtype='str')
Categorical features: Index(['Sex', 'Embarked', 'cabin_letter'], dtype='str')


In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough",
)

full_pipeline = Pipeline(
    steps=[("preprocessor", preprocessor),
           ("classifier", RandomForestClassifier())]
)

param_distributions = {
    "classifier__n_estimators": [45, 50, 55,100,200,300,500,750],
    "classifier__max_depth": [2,4, 5, 6,8,12,None],
    "classifier__min_samples_split": [2,5,9, 10,11,20],
    "classifier__min_samples_leaf": [1, 2, 4,8],
    "classifier__random_state": [1],
}

grid_search = RandomizedSearchCV(
    estimator=full_pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    cv=5,
    scoring={"accuracy": "accuracy","roc_auc": "roc_auc"},
    refit="accuracy",
    return_train_score=True,
    n_jobs=-1,
    random_state=1
)

grid_search.fit(X_train, y_train)

print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Train set - Mean CV accuracy: {results['mean_train_accuracy'][best_index]:.4f}")
print(f"Validation set - Mean CV accuracy: {results['mean_test_accuracy'][best_index]:.4f}")
print(f"Train set - Mean AUC ROC accuracy: {results['mean_train_roc_auc'][best_index]:.4f}")
print(f"Validation set - Mean AUC ROC accuracy: {results['mean_test_roc_auc'][best_index]:.4f}")

# 
final_preds = grid_search.predict(X_test)
submission = pd.DataFrame({"PassengerId": df["test"]["PassengerId"], "Survived": final_preds})
print(submission['Survived'].value_counts())
submission.to_csv(rf"{test_output_dir}/3_more_features_rf_submission.csv", index=False)

Best Parameters Found: {'classifier__random_state': 1, 'classifier__n_estimators': 750, 'classifier__min_samples_split': 20, 'classifier__min_samples_leaf': 2, 'classifier__max_depth': 12}
Train set - Mean CV accuracy: 0.7755
Validation set - Mean CV accuracy: 0.7251
Train set - Mean AUC ROC accuracy: 0.8438
Validation set - Mean AUC ROC accuracy: 0.7562
Survived
0    283
1    135
Name: count, dtype: int64


**Farily improved - 0.763 - now try lightGBM instead**

In [21]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough",
)


full_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            lgb.LGBMClassifier(verbosity=-1),
        ),
    ]
)

param_distributions = {
    "classifier__n_estimators": [100,200],
    "classifier__learning_rate": [0.02, 0.01],
    "classifier__num_leaves": [15, 31],  
    "classifier__max_depth": [3, 5, 7, -1],  
    "classifier__min_child_samples": [10,20,30,],  
    "classifier__random_state": [1],
}

# Define the identical grid search runner
grid_search = RandomizedSearchCV(
    estimator=full_pipeline,
    param_distributions=param_distributions,
    n_iter=50,
    cv=5,
    scoring={"accuracy": "accuracy", "roc_auc": "roc_auc"},
    refit="accuracy",
    return_train_score=True,
    n_jobs=-1,
    random_state=1
)

# Fit the new model
grid_search.fit(X_train, y_train)


results = grid_search.cv_results_
best_index = grid_search.best_index_

print(f"Best Parameters Found: {grid_search.best_params_}")
print(f"Train set - Mean CV accuracy: {results['mean_train_accuracy'][best_index]:.4f}")
print(f"Validation set - Mean CV accuracy: {results['mean_test_accuracy'][best_index]:.4f}")
print(f"Train set - Mean AUC ROC accuracy: {results['mean_train_roc_auc'][best_index]:.4f}")
print(f"Validation set - Mean AUC ROC accuracy: {results['mean_test_roc_auc'][best_index]:.4f}")

# Generate LightGBM Predictions
final_preds = grid_search.predict(X_test)
submission = pd.DataFrame(
    {"PassengerId": df["test"]["PassengerId"], "Survived": final_preds}
)
print(submission["Survived"].value_counts())
submission.to_csv(rf"{test_output_dir}/4_lightgbm_submission.csv", index=False)

Best Parameters Found: {'classifier__random_state': 1, 'classifier__num_leaves': 31, 'classifier__n_estimators': 200, 'classifier__min_child_samples': 10, 'classifier__max_depth': 7, 'classifier__learning_rate': 0.02}
Train set - Mean CV accuracy: 0.9099
Validation set - Mean CV accuracy: 0.8451
Train set - Mean AUC ROC accuracy: 0.9694
Validation set - Mean AUC ROC accuracy: 0.8662
Survived
0    278
1    140
Name: count, dtype: int64


D:\Python\venv\main_venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


**0.76555 using "accuracy" to guide the grid search, 0.77751 using AUC ROC. Again, with a more extensive search and an aim to avoid overfitting**

In [31]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough",
)

full_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            lgb.LGBMClassifier(verbosity=-1),
        ), 
    ]
)

param_grid = {
    "classifier__n_estimators": [100,200],
    "classifier__learning_rate": [0.001,0.01, 0.005],
    "classifier__num_leaves": [3, 4,5,6, 7],
    "classifier__max_depth": [2, 3, 4, 5],  
    "classifier__min_child_samples": [30,40,50,80],  
    "classifier__random_state": [1],
}

# Define the identical grid search runner
random_search = RandomizedSearchCV(
    estimator=full_pipeline,
    param_distributions=param_grid,
    n_iter=100,
    cv=5,
    scoring={"accuracy": "accuracy", "roc_auc": "roc_auc"},
    refit="accuracy",
    return_train_score=True,
    random_state=1,
    n_jobs=-1
)

# Fit the new model
random_search.fit(X_train, y_train)

# Extract results dictionary and best tracking index
results = random_search.cv_results_
best_index = random_search.best_index_

print(f"Best Parameters Found: {random_search.best_params_}")

# Fixed the keys inside the results extraction
print(f"Train set - Mean CV accuracy: {results['mean_train_accuracy'][best_index]:.4f}")
print(f"Validation set - Mean CV accuracy: {results['mean_test_accuracy'][best_index]:.4f}")
print(f"Train set - Mean AUC ROC accuracy: {results['mean_train_roc_auc'][best_index]:.4f}")
print(f"Validation set - Mean AUC ROC accuracy: {results['mean_test_roc_auc'][best_index]:.4f}")

# Generate LightGBM Predictions
final_preds = random_search.predict(X_test)
submission = pd.DataFrame(
    {"PassengerId": df["test"]["PassengerId"], "Survived": final_preds}
)
print(submission["Survived"].value_counts())
submission.to_csv(rf"{test_output_dir}/5_lightgbm_extensive_submission.csv", index=False)

Best Parameters Found: {'classifier__random_state': 1, 'classifier__num_leaves': 6, 'classifier__n_estimators': 200, 'classifier__min_child_samples': 40, 'classifier__max_depth': 5, 'classifier__learning_rate': 0.01}
Train set - Mean CV accuracy: 0.8322
Validation set - Mean CV accuracy: 0.8137
Train set - Mean AUC ROC accuracy: 0.8859
Validation set - Mean AUC ROC accuracy: 0.8599
Survived
0    291
1    127
Name: count, dtype: int64


D:\Python\venv\main_venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


**0.77990 using "accuracy", 0.77751 using AUC ROC. Will try a little more feature engineering and a larger search for final submission.**
